In [ ]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 8: Time Series Forecasting
==================================================================
Purpose: Forecast key business metrics (GMV, orders, revenue) using
time series models to support planning, budgeting, and resource allocation.

Key Questions:
1. What is the GMV forecast for the next quarter?
2. What are the seasonal patterns in order volume?
3. Is there a trend in delivery times or cancellation rates?
4. How can we detect anomalies in daily metrics?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE TIME SERIES FORECASTING")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
cities = pd.read_csv('../data/cities.csv')

In [ ]:
# Convert dates
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])
users['signup_date'] = pd.to_datetime(users['signup_date'])

In [ ]:
# Filter delivered orders
delivered_orders = orders[orders['order_status'] == 'delivered']

In [ ]:
print(f"✅ Loaded {len(delivered_orders):,} delivered orders")
print(f"✅ Date range: {delivered_orders['order_placed_at'].min()} to {delivered_orders['order_placed_at'].max()}")

---------------------------------------------------------------------
2. PREPARE TIME SERIES DATA
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("PREPARING TIME SERIES DATA")
print("="*80)

In [ ]:
# 2.1 Daily Aggregations
print("\n🔄 Creating daily aggregates...")

In [ ]:
daily_metrics = delivered_orders.groupby(
    delivered_orders['order_placed_at'].dt.date
).agg({
    'order_id': 'count',
    'total_amount': 'sum',
    'user_id': 'nunique'
}).reset_index()

In [ ]:
daily_metrics.columns = ['date', 'orders', 'gmv', 'unique_users']

In [ ]:
# Calculate average order value
daily_metrics['aov'] = daily_metrics['gmv'] / daily_metrics['orders']

In [ ]:
# Add date features
daily_metrics['date'] = pd.to_datetime(daily_metrics['date'])
daily_metrics['day_of_week'] = daily_metrics['date'].dt.dayofweek
daily_metrics['month'] = daily_metrics['date'].dt.month
daily_metrics['quarter'] = daily_metrics['date'].dt.quarter
daily_metrics['year'] = daily_metrics['date'].dt.year
daily_metrics['day_of_year'] = daily_metrics['date'].dt.dayofyear

In [ ]:
print(f"✅ Created {len(daily_metrics)} days of data")
print(f"📊 Date range: {daily_metrics['date'].min()} to {daily_metrics['date'].max()}")

In [ ]:
# 2.2 Weekly Aggregations
print("\n🔄 Creating weekly aggregates...")

In [ ]:
weekly_metrics = delivered_orders.groupby(
    delivered_orders['order_placed_at'].dt.to_period('W')
).agg({
    'order_id': 'count',
    'total_amount': 'sum',
    'user_id': 'nunique'
}).reset_index()

In [ ]:
weekly_metrics.columns = ['week', 'orders', 'gmv', 'unique_users']
weekly_metrics['week'] = weekly_metrics['week'].dt.start_time

In [ ]:
print(f"✅ Created {len(weekly_metrics)} weeks of data")

In [ ]:
# 2.3 Monthly Aggregations
print("\n🔄 Creating monthly aggregates...")

In [ ]:
monthly_metrics = delivered_orders.groupby(
    delivered_orders['order_placed_at'].dt.to_period('M')
).agg({
    'order_id': 'count',
    'total_amount': 'sum',
    'user_id': 'nunique'
}).reset_index()

In [ ]:
monthly_metrics.columns = ['month', 'orders', 'gmv', 'unique_users']
monthly_metrics['month'] = monthly_metrics['month'].dt.start_time

In [ ]:
print(f"✅ Created {len(monthly_metrics)} months of data")

---------------------------------------------------------------------
3. EXPLORATORY TIME SERIES ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPLORATORY TIME SERIES ANALYSIS")
print("="*80)

In [ ]:
# 3.1 Plot main metrics over time
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('QuickBite Daily Metrics Trends', fontsize=16, fontweight='bold')

In [ ]:
# Orders
ax = axes[0, 0]
ax.plot(daily_metrics['date'], daily_metrics['orders'], color='#3498db', linewidth=1.5)
ax.set_title('Daily Orders')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Orders')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# Add rolling average
rolling_mean = daily_metrics['orders'].rolling(window=7, center=True).mean()
ax.plot(daily_metrics['date'], rolling_mean, color='red', linestyle='--', 
        linewidth=2, label='7-day MA')
ax.legend()

In [ ]:
# GMV
ax = axes[0, 1]
ax.plot(daily_metrics['date'], daily_metrics['gmv'], color='#2ecc71', linewidth=1.5)
ax.set_title('Daily GMV')
ax.set_xlabel('Date')
ax.set_ylabel('GMV (₹)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
rolling_gmv = daily_metrics['gmv'].rolling(window=7, center=True).mean()
ax.plot(daily_metrics['date'], rolling_gmv, color='red', linestyle='--', 
        linewidth=2, label='7-day MA')
ax.legend()

In [ ]:
# AOV
ax = axes[1, 0]
ax.plot(daily_metrics['date'], daily_metrics['aov'], color='#e67e22', linewidth=1.5)
ax.set_title('Daily Average Order Value')
ax.set_xlabel('Date')
ax.set_ylabel('AOV (₹)')
ax.tick_params(axis='x', rotation=45)

In [ ]:
rolling_aov = daily_metrics['aov'].rolling(window=7, center=True).mean()
ax.plot(daily_metrics['date'], rolling_aov, color='red', linestyle='--', 
        linewidth=2, label='7-day MA')
ax.legend()

In [ ]:
# Unique Users
ax = axes[1, 1]
ax.plot(daily_metrics['date'], daily_metrics['unique_users'], color='#9b59b6', linewidth=1.5)
ax.set_title('Daily Unique Users')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Users')
ax.tick_params(axis='x', rotation=45)

In [ ]:
rolling_users = daily_metrics['unique_users'].rolling(window=7, center=True).mean()
ax.plot(daily_metrics['date'], rolling_users, color='red', linestyle='--', 
        linewidth=2, label='7-day MA')
ax.legend()

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/time_series_trends.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3.2 Seasonal Patterns
print("\n📊 Analyzing Seasonal Patterns...")

In [ ]:
# Day of week pattern
daily_metrics['day_name'] = daily_metrics['date'].dt.day_name()
dow_pattern = daily_metrics.groupby('day_name')['orders'].mean().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Seasonal Patterns', fontsize=14, fontweight='bold')

In [ ]:
# Day of week
ax = axes[0]
dow_pattern.plot(kind='bar', ax=ax, color='#3498db', alpha=0.7)
ax.set_title('Average Orders by Day of Week')
ax.set_xlabel('Day')
ax.set_ylabel('Average Orders')
ax.tick_params(axis='x', rotation=45)

In [ ]:
# Month of year
monthly_orders = daily_metrics.groupby('month')['orders'].mean()
ax = axes[1]
monthly_orders.plot(kind='bar', ax=ax, color='#2ecc71', alpha=0.7)
ax.set_title('Average Orders by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Average Orders')
ax.tick_params(axis='x', rotation=0)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/time_series_seasonality.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
4. STATIONARITY TEST
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("STATIONARITY TEST")
print("="*80)

In [ ]:
def check_stationarity(timeseries, series_name):
    """Perform Augmented Dickey-Fuller test for stationarity"""
    
    print(f"\n📊 {series_name} - Stationarity Test:")
    print("-" * 50)
    
    # Perform ADF test
    result = adfuller(timeseries.dropna())
    
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.4f}")
    print(f"  Critical Values:")
    for key, value in result[4].items():
        print(f"    {key}: {value:.4f}")
    
    if result[1] <= 0.05:
        print("  ✅ Series is stationary (reject H0)")
        return True
    else:
        print("  ❌ Series is non-stationary (fail to reject H0)")
        return False

In [ ]:
# Check stationarity for different metrics
order_series = daily_metrics.set_index('date')['orders']
gmv_series = daily_metrics.set_index('date')['gmv']

In [ ]:
check_stationarity(order_series, 'Orders')
check_stationarity(gmv_series, 'GMV')

---------------------------------------------------------------------
5. DECOMPOSITION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("TIME SERIES DECOMPOSITION")
print("="*80)

In [ ]:
def decompose_series(series, series_name, period=7):
    """Decompose time series into trend, seasonal, and residual components"""
    
    print(f"\n🔄 Decomposing {series_name}...")
    
    # Ensure we have enough data
    if len(series) < period * 2:
        print(f"  ⚠️ Not enough data for decomposition (need {period*2} points)")
        return None
    
    # Decompose
    decomposition = seasonal_decompose(series, model='additive', period=period)
    
    # Plot decomposition
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    fig.suptitle(f'{series_name} - Time Series Decomposition', fontsize=14, fontweight='bold')
    
    axes[0].plot(series, color='#3498db')
    axes[0].set_title('Original')
    axes[0].set_ylabel('Value')
    
    axes[1].plot(decomposition.trend, color='#2ecc71')
    axes[1].set_title('Trend')
    axes[1].set_ylabel('Value')
    
    axes[2].plot(decomposition.seasonal, color='#e67e22')
    axes[2].set_title('Seasonal')
    axes[2].set_ylabel('Value')
    
    axes[3].plot(decomposition.resid, color='#e74c3c')
    axes[3].set_title('Residual')
    axes[3].set_ylabel('Value')
    axes[3].set_xlabel('Date')
    
    plt.tight_layout()
    plt.savefig(f'../outputs/visualizations/decomposition_{series_name.lower()}.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    return decomposition

In [ ]:
# Decompose order series
order_decomp = decompose_series(order_series, 'Orders', period=7)

---------------------------------------------------------------------
6. FORECASTING WITH PROPHET
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FORECASTING WITH PROPHET")
print("="*80)

In [ ]:
try:
    # Prepare data for Prophet
    prophet_df = daily_metrics[['date', 'orders']].copy()
    prophet_df.columns = ['ds', 'y']
    
    print("\n🔄 Training Prophet model...")
    
    # Train Prophet model
    model = prophet.Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0
    )
    
    # Add country holidays (if needed)
    # model.add_country_holidays(country_name='IN')
    
    model.fit(prophet_df)
    
    # Create future dataframe
    future_days = 90
    future = model.make_future_dataframe(periods=future_days)
    
    # Forecast
    forecast = model.predict(future)
    
    print(f"✅ Prophet model trained successfully")
    print(f"📊 Forecasted {future_days} days into the future")
    
    # Plot forecast
    fig = model.plot(forecast, figsize=(14, 6))
    plt.title('Order Volume Forecast (Prophet)', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Number of Orders')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/prophet_forecast_orders.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Plot components
    fig2 = model.plot_components(forecast, figsize=(14, 8))
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/prophet_components.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Extract forecast values
    forecast_summary = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(future_days)
    forecast_summary.columns = ['date', 'forecast', 'lower_bound', 'upper_bound']
    
    print("\n📊 Next 7 Days Forecast:")
    print(forecast_summary.head(7).to_string(index=False))

In [ ]:
except Exception as e:
    print(f"⚠️ Prophet error: {e}")
    print("Using alternative forecasting method...")
    forecast_summary = None

---------------------------------------------------------------------
7. FORECASTING WITH ARIMA
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FORECASTING WITH ARIMA")
print("="*80)

In [ ]:
# Fit ARIMA model
print("\n🔄 Training ARIMA model...")

In [ ]:
try:
    # Get order series
    order_series = daily_metrics.set_index('date')['orders']
    
    # Fit ARIMA model
    model_arima = ARIMA(order_series, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7))
    model_fit = model_arima.fit()
    
    print(f"✅ ARIMA model trained successfully")
    print(f"  AIC: {model_fit.aic:.2f}")
    print(f"  BIC: {model_fit.bic:.2f}")
    
    # Forecast
    forecast_steps = 30
    forecast_arima = model_fit.forecast(steps=forecast_steps)
    
    # Create forecast dataframe
    last_date = daily_metrics['date'].max()
    future_dates = [last_date + timedelta(days=i+1) for i in range(forecast_steps)]
    
    forecast_arima_df = pd.DataFrame({
        'date': future_dates,
        'forecast': forecast_arima
    })
    
    print("\n📊 Next 7 Days Forecast (ARIMA):")
    print(forecast_arima_df.head(7).to_string(index=False))
    
    # Plot ARIMA forecast
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Historical data
    ax.plot(daily_metrics['date'], daily_metrics['orders'], 
            color='#3498db', label='Historical', linewidth=1.5)
    
    # Forecast
    ax.plot(forecast_arima_df['date'], forecast_arima_df['forecast'], 
            color='#e74c3c', label='Forecast', linewidth=2)
    
    # Confidence interval (approximate)
    std_error = np.std(model_fit.resid)
    ax.fill_between(forecast_arima_df['date'],
                    forecast_arima_df['forecast'] - 1.96 * std_error,
                    forecast_arima_df['forecast'] + 1.96 * std_error,
                    alpha=0.2, color='#e74c3c')
    
    ax.set_title('Order Volume Forecast (ARIMA)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of Orders')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/arima_forecast_orders.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
except Exception as e:
    print(f"⚠️ ARIMA error: {e}")
    forecast_arima_df = None

---------------------------------------------------------------------
8. FORECASTING WITH EXPONENTIAL SMOOTHING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FORECASTING WITH EXPONENTIAL SMOOTHING")
print("="*80)

In [ ]:
print("\n🔄 Training Exponential Smoothing model...")

In [ ]:
try:
    # Fit Holt-Winters model
    model_hw = ExponentialSmoothing(
        order_series,
        trend='add',
        seasonal='add',
        seasonal_periods=7,
        initialization_method='estimated'
    )
    
    model_hw_fit = model_hw.fit()
    
    print(f"✅ Exponential Smoothing model trained successfully")
    print(f"  AIC: {model_hw_fit.aic:.2f}")
    print(f"  BIC: {model_hw_fit.bic:.2f}")
    
    # Forecast
    forecast_steps = 30
    forecast_hw = model_hw_fit.forecast(steps=forecast_steps)
    
    # Create forecast dataframe
    last_date = daily_metrics['date'].max()
    future_dates = [last_date + timedelta(days=i+1) for i in range(forecast_steps)]
    
    forecast_hw_df = pd.DataFrame({
        'date': future_dates,
        'forecast': forecast_hw
    })
    
    print("\n📊 Next 7 Days Forecast (Exponential Smoothing):")
    print(forecast_hw_df.head(7).to_string(index=False))
    
    # Plot forecast
    fig, ax = plt.subplots(figsize=(14, 6))
    
    # Historical data
    ax.plot(daily_metrics['date'], daily_metrics['orders'], 
            color='#3498db', label='Historical', linewidth=1.5)
    
    # Forecast
    ax.plot(forecast_hw_df['date'], forecast_hw_df['forecast'], 
            color='#2ecc71', label='Forecast', linewidth=2)
    
    ax.set_title('Order Volume Forecast (Exponential Smoothing)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Number of Orders')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/hw_forecast_orders.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
except Exception as e:
    print(f"⚠️ Exponential Smoothing error: {e}")
    forecast_hw_df = None

---------------------------------------------------------------------
9. MODEL COMPARISON
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

In [ ]:
# Split data into train and test
split_date = daily_metrics['date'].quantile(0.8)
train = daily_metrics[daily_metrics['date'] < split_date]
test = daily_metrics[daily_metrics['date'] >= split_date]

In [ ]:
print(f"📊 Train size: {len(train)} days")
print(f"📊 Test size: {len(test)} days")

In [ ]:
def evaluate_forecast(y_true, y_pred):
    """Calculate forecast accuracy metrics"""
    if len(y_true) != len(y_pred):
        min_len = min(len(y_true), len(y_pred))
        y_true = y_true[:min_len]
        y_pred = y_pred[:min_len]
    
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

In [ ]:
# Evaluate models (if they exist)
model_comparison = []

In [ ]:
# For ARIMA
if forecast_arima_df is not None:
    try:
        # Get matching dates
        arima_pred = forecast_arima_df[forecast_arima_df['date'].isin(test['date'])]
        arima_metrics = evaluate_forecast(test['orders'].head(len(arima_pred)), 
                                         arima_pred['forecast'])
        model_comparison.append({
            'Model': 'ARIMA',
            'MAE': arima_metrics['MAE'],
            'RMSE': arima_metrics['RMSE'],
            'MAPE': arima_metrics['MAPE']
        })
    except:
        pass

In [ ]:
# For Exponential Smoothing
if forecast_hw_df is not None:
    try:
        hw_pred = forecast_hw_df[forecast_hw_df['date'].isin(test['date'])]
        hw_metrics = evaluate_forecast(test['orders'].head(len(hw_pred)), 
                                      hw_pred['forecast'])
        model_comparison.append({
            'Model': 'Exponential Smoothing',
            'MAE': hw_metrics['MAE'],
            'RMSE': hw_metrics['RMSE'],
            'MAPE': hw_metrics['MAPE']
        })
    except:
        pass

In [ ]:
# Simple baseline (moving average)
baseline_forecast = test['orders'].rolling(window=7).mean().shift(1).fillna(test['orders'].mean())
baseline_metrics = evaluate_forecast(test['orders'], baseline_forecast)
model_comparison.append({
    'Model': '7-day Moving Average',
    'MAE': baseline_metrics['MAE'],
    'RMSE': baseline_metrics['RMSE'],
    'MAPE': baseline_metrics['MAPE']
})

In [ ]:
model_comparison_df = pd.DataFrame(model_comparison)
print("\n📊 Model Comparison:")
print(model_comparison_df.to_string(index=False))

---------------------------------------------------------------------
10. ANOMALY DETECTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("ANOMALY DETECTION")
print("="*80)

In [ ]:
def detect_anomalies(series, window=14, threshold=2.5):
    """
    Detect anomalies using rolling statistics
    """
    rolling_mean = series.rolling(window=window, center=True).mean()
    rolling_std = series.rolling(window=window, center=True).std()
    
    upper_bound = rolling_mean + threshold * rolling_std
    lower_bound = rolling_mean - threshold * rolling_std
    
    anomalies = (series > upper_bound) | (series < lower_bound)
    
    return anomalies, upper_bound, lower_bound

In [ ]:
# Detect anomalies in orders
anomalies, upper_bound, lower_bound = detect_anomalies(
    daily_metrics.set_index('date')['orders']
)

In [ ]:
# Plot anomalies
fig, ax = plt.subplots(figsize=(14, 6))

In [ ]:
ax.plot(daily_metrics['date'], daily_metrics['orders'], 
        color='#3498db', label='Orders', linewidth=1.5)

In [ ]:
# Add bounds
ax.fill_between(daily_metrics['date'], lower_bound, upper_bound, 
                alpha=0.2, color='#2ecc71', label='Normal Range')

In [ ]:
# Highlight anomalies
anomaly_dates = daily_metrics['date'][anomalies]
anomaly_values = daily_metrics['orders'][anomalies]

In [ ]:
ax.scatter(anomaly_dates, anomaly_values, color='red', s=50, 
           label='Anomalies', zorder=5)

In [ ]:
ax.set_title('Anomaly Detection in Daily Orders', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Orders')
ax.legend()
ax.grid(True, alpha=0.3)
ax.tick_params(axis='x', rotation=45)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/anomaly_detection.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(f"\n📊 Detected {anomalies.sum()} anomalies in daily orders")

---------------------------------------------------------------------
11. FORECASTING BUSINESS METRICS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("FORECASTING BUSINESS METRICS")
print("="*80)

In [ ]:
# 11.1 GMV Forecast
print("\n🔄 Forecasting GMV...")

In [ ]:
gmv_series = daily_metrics.set_index('date')['gmv']

In [ ]:
try:
    # Fit ARIMA model for GMV
    model_gmv = ARIMA(gmv_series, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7))
    model_gmv_fit = model_gmv.fit()
    
    # Forecast GMV
    forecast_steps = 30
    gmv_forecast = model_gmv_fit.forecast(steps=forecast_steps)
    
    # Create forecast dataframe
    last_date = daily_metrics['date'].max()
    future_dates = [last_date + timedelta(days=i+1) for i in range(forecast_steps)]
    
    gmv_forecast_df = pd.DataFrame({
        'date': future_dates,
        'forecast_gmv': gmv_forecast
    })
    
    print("\n📊 Next 7 Days GMV Forecast:")
    print(gmv_forecast_df.head(7).to_string(index=False))
    
    # Calculate projected monthly GMV
    projected_monthly_gmv = gmv_forecast_df['forecast_gmv'].sum()
    print(f"\n📊 Projected GMV for next 30 days: ₹{projected_monthly_gmv:,.2f}")

In [ ]:
except Exception as e:
    print(f"⚠️ GMV forecasting error: {e}")

In [ ]:
# 11.2 User Growth Forecast
print("\n🔄 Forecasting User Growth...")

In [ ]:
user_series = daily_metrics.set_index('date')['unique_users']

In [ ]:
try:
    # Fit ARIMA model for users
    model_users = ARIMA(user_series, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7))
    model_users_fit = model_users.fit()
    
    # Forecast users
    forecast_steps = 30
    user_forecast = model_users_fit.forecast(steps=forecast_steps)
    
    # Create forecast dataframe
    last_date = daily_metrics['date'].max()
    future_dates = [last_date + timedelta(days=i+1) for i in range(forecast_steps)]
    
    user_forecast_df = pd.DataFrame({
        'date': future_dates,
        'forecast_users': user_forecast
    })
    
    print("\n📊 Next 7 Days User Forecast:")
    print(user_forecast_df.head(7).to_string(index=False))

In [ ]:
except Exception as e:
    print(f"⚠️ User forecasting error: {e}")

---------------------------------------------------------------------
12. BUSINESS RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("BUSINESS RECOMMENDATIONS")
print("="*80)

In [ ]:
# Calculate trends
recent_avg = daily_metrics['orders'].tail(30).mean()
previous_avg = daily_metrics['orders'].tail(60).head(30).mean()
trend_pct = ((recent_avg - previous_avg) / previous_avg) * 100

In [ ]:
print(f"""
🏆 KEY INSIGHTS & RECOMMENDATIONS:
==================================

1. TREND ANALYSIS:
   • Current order volume: {recent_avg:.0f} orders/day
   • Previous period: {previous_avg:.0f} orders/day
   • Trend: {trend_pct:+.1f}% change
   • Seasonality: {dow_pattern.max():.0f} orders on peak days vs {dow_pattern.min():.0f} on trough days

2. SEASONAL PATTERNS:
   • Best day: {dow_pattern.idxmax()} ({dow_pattern.max():.0f} avg orders)
   • Worst day: {dow_pattern.idxmin()} ({dow_pattern.min():.0f} avg orders)
   • Weekend effect: {dow_pattern[['Saturday', 'Sunday']].mean():.0f} orders
   • Weekday effect: {dow_pattern[['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']].mean():.0f} orders

3. FORECAST INSIGHTS:
   • Projected next 30 days: {projected_monthly_gmv:,.0f} GMV
   • Expected daily orders: {gmv_forecast_df['forecast_gmv'].mean() / daily_metrics['aov'].mean():.0f} orders
   • Growth trajectory: {'Increasing' if trend_pct > 5 else 'Stable' if trend_pct > -5 else 'Declining'}

4. ANOMALY DETECTION:
   • {anomalies.sum()} anomalies detected in daily orders
   • Most recent anomaly: {anomaly_dates.max() if len(anomaly_dates) > 0 else 'None'}
   • Investigation needed for significant deviations

🎯 ACTIONABLE RECOMMENDATIONS:
==============================

PRIORITY 1 (Immediate - Next 30 Days):
---------------------------------------
1. Prepare for weekend demand spikes:
   • Ensure adequate delivery partner supply
   • Restaurant capacity planning
   • Marketing campaigns aligned with peak days

2. Investigate anomalies:
   • Root cause analysis for unusual spikes/dips
   • Operational issue identification
   • Data quality validation

3. Resource allocation based on forecast:
   • Staff scheduling aligned with predicted demand
   • Inventory planning for restaurant partners
   • Marketing spend optimization

PRIORITY 2 (Short-term - Next 90 Days):
--------------------------------------
1. Implement automated anomaly detection:
   • Real-time monitoring dashboard
   • Alert system for significant deviations
   • Automated investigation workflows

2. Optimize for seasonality:
   • Day-of-week specific promotions
   • Monthly campaign planning
   • Holiday preparation

3. Improve forecasting accuracy:
   • Incorporate external factors (weather, events)
   • Ensemble modeling approach
   • Regular model retraining

PRIORITY 3 (Long-term - Next 6 Months):
--------------------------------------
1. Build predictive analytics platform:
   • Real-time forecasting capabilities
   • Integration with operational systems
   • Automated decision support

2. Develop leading indicators:
   • Early warning system for demand changes
   • Predictive maintenance for supply chain
   • Proactive customer retention

📈 SUCCESS METRICS:
==================
• Forecasting accuracy (MAPE < 10%)
• Anomaly detection rate (> 90%)
• Operational efficiency improvement (15%)
• Resource utilization optimization (20%)
""")

---------------------------------------------------------------------
13. EXPORT RESULTS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("EXPORTING RESULTS")
print("="*80)

In [ ]:
# Save daily metrics with anomalies
daily_metrics['is_anomaly'] = anomalies
daily_metrics.to_csv('../outputs/cleaned_data/daily_metrics_with_anomalies.csv', index=False)

In [ ]:
# Save forecasts if available
if forecast_summary is not None:
    forecast_summary.to_csv('../outputs/cleaned_data/prophet_forecast_orders.csv', index=False)

In [ ]:
if forecast_arima_df is not None:
    forecast_arima_df.to_csv('../outputs/cleaned_data/arima_forecast_orders.csv', index=False)

In [ ]:
if gmv_forecast_df is not None:
    gmv_forecast_df.to_csv('../outputs/cleaned_data/gmv_forecast.csv', index=False)

In [ ]:
print("✅ Daily metrics with anomalies saved")
print("✅ Forecast data saved")
print("✅ Visualizations saved to ../outputs/visualizations/")

---------------------------------------------------------------------
14. FINAL SUMMARY
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("TIME SERIES FORECASTING - SUMMARY")
print("="*80)

In [ ]:
print(f"""
📊 TIME SERIES ANALYSIS SUMMARY:
===============================

DATA OVERVIEW:
• Analysis period: {daily_metrics['date'].min()} to {daily_metrics['date'].max()}
• Total days: {len(daily_metrics):,}
• Total weeks: {len(weekly_metrics):,}
• Total months: {len(monthly_metrics):,}

METRICS SUMMARY:
• Average daily orders: {daily_metrics['orders'].mean():.0f}
• Average daily GMV: ₹{daily_metrics['gmv'].mean():,.2f}
• Average daily users: {daily_metrics['unique_users'].mean():.0f}
• Average order value: ₹{daily_metrics['aov'].mean():.2f}

FORECAST PERFORMANCE:
• Best performing model: {model_comparison_df.iloc[model_comparison_df['MAPE'].idxmin()]['Model']}
• Best MAPE: {model_comparison_df['MAPE'].min():.2f}%
• Best RMSE: {model_comparison_df['RMSE'].min():.2f}

SEASONALITY:
• Weekly pattern: {dow_pattern.max() - dow_pattern.min():.0f} order variance
• Peak day: {dow_pattern.idxmax()}
• Trough day: {dow_pattern.idxmin()}

ANOMALIES:
• Total anomalies detected: {anomalies.sum()}
• Anomaly rate: {anomalies.sum() / len(anomalies) * 100:.2f}%
• Most recent anomaly: {anomaly_dates.max() if len(anomaly_dates) > 0 else 'None'}
""")

In [ ]:
print("\n" + "="*80)
print("✅ TIME SERIES FORECASTING COMPLETE")
print("="*80)
print("\n📌 Next Steps:")
print("  1. Implement forecasting models in production")
print("  2. Set up automated anomaly detection")
print("  3. Integrate forecasts with operational planning")
print("  4. Monitor forecast accuracy and retrain models")
print("="*80)